In [ ]:
import requests
import pandas as pd
import re
import time
from tqdm import tqdm
from google.colab import files

##############################################################
# USER SETTINGS
##############################################################

EMAIL = "simon.anguzu@students.mak.ac.ug"

# Category 1: Bee Species / Taxa (Supports space variations like "bumble bee")
BEE_REGEX = re.compile(
    r'(honey\s*bee|honeybee|apis|apis\s+mellifera|apis\s+cerana|apis\s+dorsata|'
    r'stingless\s+bee|meliponini|melipona|meliponula|tetragonisca|bumble\s*bee|bumblebee|bombus|'
    r'bombus\s+terrestris|bombus\s+impatiens|non-apis|wild\s+bee|apoidea)',
    re.IGNORECASE
)

# Category 2: Agrochemicals / Pesticides / Specific Chemicals
PESTICIDE_REGEX = re.compile(
    r'(pesticide|agrochemical|insecticide|neonicotinoid|imidacloprid|thiamethoxam|'
    r'clothianidin|acetamiprid|thiacloprid|flupyradifurone|fungicide|herbicide|'
    r'glyphosate|acaricide|fipronil|organophosphate|pyrethroid|carbamate|'
    r'chlorpyrifos|deltamethrin)',
    re.IGNORECASE
)

# Category 3: Omics / Gene Expression Technologies
OMICS_REGEX = re.compile(
    r'(transcriptom|rna\s*-?\s*seq|rna\s+sequencing|gene\s+expression|differentially\s+expressed|'
    r'deg|degs|microarray|proteom|metabolom|multi-omics|multiomics|ecotoxicogenom|'
    r'toxicogenom|next\s+generation\s+sequencing|ngs)',
    re.IGNORECASE
)

##############################################################
# CROSSREF API FETCH (USING MULTI-QUERY SEARCH)
##############################################################

url = "https://api.crossref.org/works"

headers = {
    "User-Agent": f"ComprehensiveBeeToxicogenomics/3.0 (mailto:{EMAIL})"
}

# Broad query targeting all key areas simultaneously
params = {
    "query": "bee imidacloprid neonicotinoid pesticide gene expression transcriptome",
    "filter": "from-pub-date:2010-01-01,until-pub-date:2026-12-31,type:journal-article",
    "rows": 100,
    "offset": 0
}

print("Searching Crossref repository...")
response = requests.get(url, headers=headers, params=params)
response.raise_for_status()

total_candidates = response.json()["message"]["total-results"]
print(f"Total candidate articles retrieved for screening: {total_candidates}\n")

##############################################################
# FLEXIBLE THREE-WAY CATEGORY FILTERING
##############################################################

filtered_records = []
batch_size = 100
max_candidates = min(total_candidates, 5000)  # Expanded candidate window

for offset in tqdm(range(0, max_candidates, batch_size), desc="Extracting & Validating Papers"):
    params["offset"] = offset
    resp = requests.get(url, headers=headers, params=params)

    if resp.status_code == 200:
        items = resp.json().get("message", {}).get("items", [])

        for item in items:
            title = item.get("title", [""])[0] if item.get("title") else ""
            abstract = item.get("abstract", "")

            # Combine Title + Abstract for strict evaluation
            combined_text = f"{title} {abstract}"

            # Fallback evaluation: Check if matches Bees AND (Pesticides OR Omics) in Title/Abstract
            has_bee = bool(BEE_REGEX.search(combined_text))
            has_pesticide = bool(PESTICIDE_REGEX.search(combined_text))
            has_omics = bool(OMICS_REGEX.search(combined_text))

            # Save if matches all 3 categories OR matches Bee + (Pesticide/Omics) in Title
            if (has_bee and has_pesticide and has_omics) or (has_bee and (has_pesticide or has_omics)):

                pub_year = ""
                if "published" in item and "date-parts" in item["published"]:
                    pub_year = item["published"]["date-parts"][0][0]
                elif "published-print" in item and "date-parts" in item["published-print"]:
                    pub_year = item["published-print"]["date-parts"][0][0]

                journal = item.get("container-title", [""])[0] if item.get("container-title") else ""

                authors = []
                for author in item.get("author", []):
                    given = author.get("given", "")
                    family = author.get("family", "")
                    authors.append(f"{given} {family}".strip())
                author_str = "; ".join(authors)

                filtered_records.append({
                    "DOI": item.get("DOI", ""),
                    "Title": title,
                    "Authors": author_str,
                    "Journal": journal,
                    "Year": pub_year,
                    "Publisher": item.get("publisher", ""),
                    "URL": item.get("URL", "")
                })

    time.sleep(0.2)

##############################################################
# SAVE & EXPORT
##############################################################

df = pd.DataFrame(filtered_records)
df = df.drop_duplicates(subset=["DOI"], keep="first")

filename = "Complete_Bee_Toxicogenomics_Crossref_2010_2026.xlsx"
df.to_excel(filename, index=False, engine='openpyxl')

print(f"\nExtraction Complete!")
print(f"Total papers extracted: {len(df)}")
print(df.head(3))

# Auto-download in Google Colab
files.download(filename)

Searching Crossref repository...
Total candidate articles retrieved for screening: 877492



Extracting & Validating Papers: 100%|██████████| 50/50 [01:54<00:00,  2.29s/it]


Extraction Complete!
Total papers extracted: 406
                              DOI  \
0              10.1242/jeb.127472   
1  10.1016/j.jinsphys.2016.03.004   
2              10.1242/jeb.230060   

                                               Title  \
0  A neonicotinoid, imidacloprid, impairs honey b...   
1  Sperm viability and gene expression in honey b...   
2  Correction: The neonicotinoid imidacloprid imp...   

                                             Authors  \
0                         Erica Zhang; James C. Nieh   
1  Veeranan Chaimanee; Jay D. Evans; Yanping Chen...   
2                         Erica Zhang; James C. Nieh   

                           Journal  Year                  Publisher  \
0  Journal of Experimental Biology  2015  The Company of Biologists   
1     Journal of Insect Physiology  2016                Elsevier BV   
2  Journal of Experimental Biology  2020  The Company of Biologists   

                                              URL  
0             

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>